# 1970 songs: preprocessing, exploration, features and communities

Run the cells from top to bottom. First generate `songs_1970.csv` and `genius_songs_trimmed.csv` using the existing download scripts. The analysis outputs are written to `analysis/`.

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data_exploration.py').exists():
    ROOT = next(parent for parent in [ROOT, *ROOT.parents] if (parent / 'data_exploration.py').exists())
DATA_DIR = next((folder for folder in (ROOT / 'Data', ROOT / 'data', ROOT) if folder.exists()), ROOT)
sys.path.insert(0, str(ROOT))
SONGS_PATH = DATA_DIR / 'songs_1970.csv'
RAW_LYRICS_PATH = DATA_DIR / 'genius_songs_trimmed.csv'
LYRICS_PATH = DATA_DIR / 'song_lyrics_1970.csv'
OUTPUT_PATH = ROOT / 'analysis'
OUTPUT_PATH.mkdir(exist_ok=True)
print(f'Project root: {ROOT}')
print(f'Data folder: {DATA_DIR}')
print(f'Songs available: {SONGS_PATH.exists()}')
print(f'Lyrics available: {RAW_LYRICS_PATH.exists()}')

Project root: c:\Users\cheli\OneDrive\Dokumenter\GitHub\SocialGraphs_FinalProject
Data folder: c:\Users\cheli\OneDrive\Dokumenter\GitHub\SocialGraphs_FinalProject\Data
Songs available: True
Lyrics available: True


## 1. Match and preprocess lyrics

This keeps only lyrics whose normalized title and artist occur in the MusicBrainz song table. Normalization is case-insensitive and removes bracketed annotations such as `[Remastered]`.

In [3]:
from filter_song_lyrics import filter_lyrics

if not SONGS_PATH.exists() or not RAW_LYRICS_PATH.exists():
    print('Generate songs_1970.csv and genius_songs_trimmed.csv before running this cell.')
else:
    LYRICS_PATH.parent.mkdir(exist_ok=True)
    matched = filter_lyrics(SONGS_PATH, RAW_LYRICS_PATH, LYRICS_PATH)
    print(f'Matched lyric rows: {matched}')

Matched lyric rows: 338


## 2. Load data and inspect quality

In [5]:
import pandas as pd
from data_exploration import load_songs, load_lyrics, clean_lyrics

songs = load_songs(SONGS_PATH)
lyrics = load_lyrics(LYRICS_PATH)
songs = songs.merge(
    lyrics[['song_key', 'artist_key', 'lyrics_clean']],
    on=['song_key', 'artist_key'], how='left'
)
songs['lyrics_clean'] = songs['lyrics_clean'].fillna('')
display(songs.head())
display(pd.DataFrame({
    'rows': [len(songs)],
    'unique_songs': [songs['id'].nunique()],
    'unique_artists': [songs['artist_key'].nunique()],
    'missing_lyrics': [(songs['lyrics_clean'] == '').sum()],
    'duplicate_ids': [songs['id'].duplicated().sum()],
}))

,id,song,artist,genre,year,language,country,genres,song_key,artist_key,genre_list,lyrics_clean
0,6cca67b4-35ab-486e-8525-0665f2d67aee,Spanish Key (single edit),Miles Davis,jazz,1970,eng,US,jazz;jazz fusion;modal jazz;cool jazz;hard bop...,spanish key,miles davis,"[bebop, blues, classical, cool jazz, funk, har...",
1,130897fc-2139-4a68-8fcd-88b28abe4438,I Can’t Find Her,The Rotations,funk,1970,eng,US,funk;soul,i can t find her,the rotations,"[funk, soul]",
2,718692c5-ae33-4497-96dc-95f0578e20ca,I’ll Never Forget You,The Jaggerz,pop rock,1970,eng,US,pop rock;blue-eyed soul,i ll never forget you,the jaggerz,"[blue-eyed soul, pop rock]",
3,03a56a3f-ecfc-4bb7-83a1-b967d595cb4a,"Yeah, You Right",Sisters and Brothers,,1970,eng,US,,yeah you right,sisters and brothers,[],
4,370c166a-d198-417b-831a-511bb6c206a4,Lincoln Street Chapel,Vicki Lawrence,pop,1970,eng,US,pop,lincoln street chapel,vicki lawrence,[pop],


,rows,unique_songs,unique_artists,missing_lyrics,duplicate_ids
0,2013,2013,829,1675,0


## 3. Visualize the dataset

In [6]:
from data_exploration import write_exploration_plots

write_exploration_plots(songs, OUTPUT_PATH)
display(songs['genre'].replace('', 'Unknown').value_counts().head(15).to_frame('songs'))
display(songs['country'].replace('', 'Unknown').value_counts().head(15).to_frame('songs'))
display(songs['lyrics_clean'].str.split().str.len().describe().to_frame('words'))
print(f'Plots written to {OUTPUT_PATH}')

,songs
genre,
Unknown,783
rock,212
soul,174
pop,135
country,126
psychedelic rock,52
funk,50
progressive rock,50
jazz,48


,songs
country,
US,836
GB,316
NL,247
DE,135
FR,82
IE,78
CA,66
AU,48
JM,36


,words
count,2013.000000
mean,34.442623
std,84.038530
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,546.000000


Plots written to c:\Users\cheli\OneDrive\Dokumenter\GitHub\SocialGraphs_FinalProject\analysis


## 4. Build features and cluster songs

Features include multi-label genres, language/country metadata, and lyric TF-IDF unigrams/bigrams. K-means is used as an initial baseline; inspect the resulting groups rather than treating the number of clusters as final.

In [7]:
from data_exploration import build_features

build_features(songs, OUTPUT_PATH)
clusters = pd.read_csv(OUTPUT_PATH / 'song_clusters.csv') if (OUTPUT_PATH / 'song_clusters.csv').exists() else pd.DataFrame()
if not clusters.empty:
    display(clusters.groupby('cluster').size().rename('songs').to_frame())
    display(clusters.merge(songs[['id', 'genre']], on='id').sort_values('cluster').head(20))
else:
    print('Not enough usable rows/features for K-means.')

c:\Users\cheli\miniforge3\envs\Temp\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=8.
  warnings.warn(


,songs
cluster,
0,470
1,217
2,177
3,118
4,247
5,152
6,489
7,143


,id,song,artist,cluster,genre
1995,4fce69ef-957b-4234-ad84-c7b6701e7155,Take Me In Your Arms,Stern Selection,0,
1996,d6674e83-2925-4b40-b3f3-db11b5fea073,Make Love Not War,John Woolley & Just Born,0,
1997,8039bd2e-0db3-41a5-bfa4-c52d6142aed9,Ruby Baby,John Woolley & Just Born,0,
2000,e3a3a3aa-c476-421b-94a1-2d86c7b26896,Take a Ride in the Sky,Socrates Drank the Conium,0,progressive rock
2002,62cc3811-48f6-4361-a0a3-c011b663288b,Sad Memories,Stern Selection,0,
2004,8661929b-f94f-44ad-86f7-440fc9ed1fb2,Return to Italy,Eric,0,
2005,00bd88f9-9257-42a8-b29f-50c66dbe5fb6,Good Days Have Gone,Demis Roussos,0,
2012,eebb9ff5-aeda-4259-b2b7-437e193894a1,Walking 'round,Eric Spitzer-Marlyn,0,
2011,e6de875c-e8ca-498b-b4d6-411c56b552d4,Hot Pants,Salvage,0,
2010,4801f245-8dd4-4e73-865d-29a660cd161b,The Foundation of Love,Salvage,0,


## 5. Build the first community model

The graph links songs that share an artist or genre. Greedy modularity detects communities in the weighted song projection. The GEXF file can be opened in Gephi for further inspection.

In [8]:
from data_exploration import build_communities

build_communities(songs, OUTPUT_PATH)
communities = pd.read_csv(OUTPUT_PATH / 'song_communities.csv')
display(communities.groupby('community').agg(songs=('id', 'size'), mean_degree=('degree', 'mean')).sort_values('songs', ascending=False).head(20))
display(communities.merge(songs[['id', 'song', 'artist', 'genre']], on='id').sort_values(['community', 'degree'], ascending=[True, False]).head(30))

Community graph: 2013 songs, 230646 edges, 346 communities


,songs,mean_degree
community,,
0,712,440.543539
1,341,309.158358
2,189,211.206349
3,34,30.647059
4,9,10.777778
5,8,7.000000
6,8,7.000000
7,6,5.000000
8,6,5.000000


,id,community,degree,song,artist,genre
337,3409e640-8579-49b5-8dbb-374285ac1fa2,0,956,Take Me to the Pilot,Elton John,pop
1975,d1f14d09-a59b-4a67-b23c-4d91b7d89332,0,902,I Know I’m Losing You,Rare Earth,psychedelic rock
349,0c3def50-a98f-414a-ad42-7a1d285f6b94,0,830,United We Stand,Brotherhood of Man,pop
759,a777a660-07e8-44ae-96d1-47c34cdae54f,0,806,She Cheats On Me,Roy Orbison,rockabilly
760,1feb7a66-dea5-4474-88c2-01fd6c36fd46,0,806,How Do You Start Over?,Roy Orbison,rockabilly
79,267ec755-2087-4e94-bda8-a0894867b340,0,791,One Toke Over the Line,Brewer & Shipley,rock
935,a231668f-590c-4cce-ba2d-c1f5f786bf22,0,787,Question,The Moody Blues,progressive rock
955,e4f5defe-7326-4b4f-b36a-bcc8e3ef2028,0,787,Candle of Life,The Moody Blues,progressive rock
1107,13a7641f-7a33-4e8b-b054-9ca4c26da7c1,0,787,Melancholy Man,The Moody Blues,progressive rock
1666,5f107f42-2773-455f-9bc1-d04db3d6458f,0,772,Honey Come Back,Glen Campbell,country


## 6. Save the final cleaned table

This table is the starting point for downstream graph analysis or predictive modelling.

In [9]:
songs.to_csv(OUTPUT_PATH / 'songs_clean.csv', index=False)
print(f'Saved {len(songs)} rows to {OUTPUT_PATH / "songs_clean.csv"}')

Saved 2013 rows to c:\Users\cheli\OneDrive\Dokumenter\GitHub\SocialGraphs_FinalProject\analysis\songs_clean.csv
